In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, os
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, roc_curve, f1_score)

print("All imports successful.")

All imports successful.


In [2]:
df = pd.read_csv('D:/Shreya/SEM 2/MINOR PROJECT/upi-fraud-detection/data/upi_transactions_2024.csv')
df = df.drop(columns=['transaction id', 'timestamp', 'transaction_status'])
print("Shape after dropping:", df.shape)

Shape after dropping: (250000, 14)


In [3]:
df['is_night']       = (df['hour_of_day'] <= 4).astype(int)
df['is_high_amount'] = (df['amount (INR)'] > 1596).astype(int)
df['amount_band']    = pd.cut(df['amount (INR)'],
                               bins=[0, 500, 1500, 5000, 50000],
                               labels=[0, 1, 2, 3]).astype(int)
print("Features created.")

Features created.


In [4]:
categorical_cols = [
    'transaction type', 'merchant_category', 'sender_age_group',
    'receiver_age_group', 'sender_state', 'sender_bank',
    'receiver_bank', 'device_type', 'network_type', 'day_of_week'
]

label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le

os.makedirs('D:/Shreya/SEM 2/MINOR PROJECT/upi-fraud-detection/model', exist_ok=True)
joblib.dump(label_encoders, 'D:/Shreya/SEM 2/MINOR PROJECT/upi-fraud-detection/model/label_encoders.pkl')
print("Encoding done. Dtypes:")
print(df.dtypes)

Encoding done. Dtypes:
transaction type      int64
merchant_category     int64
amount (INR)          int64
sender_age_group      int64
receiver_age_group    int64
sender_state          int64
sender_bank           int64
receiver_bank         int64
device_type           int64
network_type          int64
fraud_flag            int64
hour_of_day           int64
day_of_week           int64
is_weekend            int64
is_night              int64
is_high_amount        int64
amount_band           int64
dtype: object


In [5]:
X = df.drop(columns=['fraud_flag'])
y = df['fraud_flag']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print("X_train:", X_train.shape, "| Fraud:", y_train.sum())
print("X_test: ", X_test.shape,  "| Fraud:", y_test.sum())

X_train: (200000, 16) | Fraud: 384
X_test:  (50000, 16) | Fraud: 96


In [6]:
scale_cols = ['amount (INR)', 'hour_of_day']
scaler = StandardScaler()
X_train[scale_cols] = scaler.fit_transform(X_train[scale_cols])
X_test[scale_cols]  = scaler.transform(X_test[scale_cols])

joblib.dump(scaler, 'D:/Shreya/SEM 2/MINOR PROJECT/upi-fraud-detection/model/scaler.pkl')
print("Scaling done.")
print(X_train[scale_cols].describe().round(3))

Scaling done.
       amount (INR)  hour_of_day
count    200000.000   200000.000
mean         -0.000       -0.000
std           1.000        1.000
min          -0.704       -2.831
25%          -0.554       -0.711
50%          -0.369        0.059
75%           0.153        0.830
max          22.037        1.601


In [7]:
print("Before SMOTE:", y_train.value_counts().to_dict())

smote = SMOTE(random_state=42, k_neighbors=5)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print("After SMOTE: ", pd.Series(y_train_sm).value_counts().to_dict())
print("X_train_sm shape:", X_train_sm.shape)

Before SMOTE: {0: 199616, 1: 384}
After SMOTE:  {0: 199616, 1: 199616}
X_train_sm shape: (399232, 16)


In [8]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    class_weight=None,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train_sm, y_train_sm)
print("Random Forest done.")

Random Forest done.


In [9]:
xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=1,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)
xgb_model.fit(X_train_sm, y_train_sm,
              eval_set=[(X_test, y_test)],
              verbose=50)
print("XGBoost done.")

[0]	validation_0-logloss:0.66671
[50]	validation_0-logloss:0.26330
[100]	validation_0-logloss:0.14920
[150]	validation_0-logloss:0.09759
[199]	validation_0-logloss:0.07602
XGBoost done.


In [10]:
def evaluate_model(model, name, X_test, y_test, threshold=0.3):
    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= threshold).astype(int)

    print(f"\n{'='*50}")
    print(f"  {name}  (threshold={threshold})")
    print(f"{'='*50}")
    print(classification_report(y_test, y_pred,
                                target_names=['Legitimate', 'Fraud']))
    auc = roc_auc_score(y_test, y_prob)
    print(f"ROC-AUC: {auc:.4f}")
    caught = int(((y_pred==1) & (y_test==1)).sum())
    total  = int((y_test==1).sum())
    print(f"Fraud caught: {caught} / {total}")
    return y_pred, y_prob

rf_pred,  rf_prob  = evaluate_model(rf_model,  "Random Forest", X_test, y_test)
xgb_pred, xgb_prob = evaluate_model(xgb_model, "XGBoost",       X_test, y_test)


  Random Forest  (threshold=0.3)
              precision    recall  f1-score   support

  Legitimate       1.00      0.59      0.75     49904
       Fraud       0.00      0.35      0.00        96

    accuracy                           0.59     50000
   macro avg       0.50      0.47      0.37     50000
weighted avg       1.00      0.59      0.74     50000

ROC-AUC: 0.4722
Fraud caught: 34 / 96

  XGBoost  (threshold=0.3)
              precision    recall  f1-score   support

  Legitimate       1.00      0.97      0.98     49904
       Fraud       0.00      0.04      0.00        96

    accuracy                           0.96     50000
   macro avg       0.50      0.50      0.49     50000
weighted avg       1.00      0.96      0.98     50000

ROC-AUC: 0.4738
Fraud caught: 4 / 96
